In [1]:
!pwd

/Users/anshulchiranth/Desktop/Strike/Voice Experiments


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("AAI_API_KEY")

In [4]:
import assemblyai as aai
aai.settings.api_key = api_key

In [20]:
from pathlib import Path
import requests
import time

base_dir = Path.cwd()

julian_dir = base_dir / "Unclipped Processed Transactions" / "Julian Files"
george_dir = base_dir / "Unclipped Processed Transactions" / "George Files"

In [6]:
julian_files = [f for f in julian_dir.iterdir()
               if f.is_file() and not f.name.startswith(".")]

george_files = [f for f in george_dir.iterdir()
               if f.is_file() and not f.name.startswith(".")]

In [21]:
base_url = "https://api.assemblyai.com"

headers = {
"authorization": api_key
}

In [30]:
#Script to collect transcript with timestamps and operator/customer splits for julian files
julian_transcripts = {}

for file in julian_files:
    with open(file, "rb") as f:
        response = requests.post(base_url + "/v2/upload", headers=headers, data=f)
        if response.status_code != 200:
            print(f"Error: {response.status_code}, Response: {response.text}")
            response.raise_for_status()
        upload_json = response.json()
        audio_file = upload_json["upload_url"]


    data = {
    "audio_url": audio_file,
    "speech_models": ["universal-3-pro", "universal-2"],
    "language_detection": True,
    "speaker_labels": True,
    "speech_understanding": {
        "request": {
            "speaker_identification": {
                "speaker_type": "name",
                "known_values": ["Customer", "Operator"]
                }
            }
        }
    }
    
    response = requests.post(base_url + "/v2/transcript", headers=headers, json=data)
    transcript_id = response.json()["id"]
    polling_endpoint = base_url + f"/v2/transcript/{transcript_id}"

    while True:
        transcript = requests.get(polling_endpoint, headers=headers).json()
        if transcript["status"] == "completed":
            break
        elif transcript["status"] == "error":
            raise RuntimeError(f"Transcription failed: {transcript['error']}")
        else:
            time.sleep(3)

    julian_transcripts[file] = transcript

In [39]:
#Script to collect transcript with timestamps and operator/customer splits for george files
george_transcripts = {}

for file in george_files:
    with open(file, "rb") as f:
        response = requests.post(base_url + "/v2/upload", headers=headers, data=f)
        if response.status_code != 200:
            print(f"Error: {response.status_code}, Response: {response.text}")
            response.raise_for_status()
        upload_json = response.json()
        audio_file = upload_json["upload_url"]


    data = {
    "audio_url": audio_file,
    "speech_models": ["universal-3-pro", "universal-2"],
    "language_detection": True,
    "speaker_labels": True,
    "speech_understanding": {
        "request": {
            "speaker_identification": {
                "speaker_type": "name",
                "known_values": ["Customer", "Operator"]
                }
            }
        }
    }
    
    response = requests.post(base_url + "/v2/transcript", headers=headers, json=data)
    transcript_id = response.json()["id"]
    polling_endpoint = base_url + f"/v2/transcript/{transcript_id}"

    while True:
        transcript = requests.get(polling_endpoint, headers=headers).json()
        if transcript["status"] == "completed":
            break
        elif transcript["status"] == "error":
            raise RuntimeError(f"Transcription failed: {transcript['error']}")
        else:
            time.sleep(3)

    george_transcripts[file] = transcript

In [55]:
julian_intervals = {}

for key, transcript in julian_transcripts.items():
    intervals = []
    for utterance in transcript["utterances"]:
        if utterance["speaker"] == "Operator":
            intervals.append((utterance["start"], utterance["end"]))
    julian_intervals[key] = intervals

In [57]:
george_intervals = {}

for key, transcript in george_transcripts.items():
    intervals = []
    for utterance in transcript["utterances"]:
        if utterance["speaker"] == "Operator":
            intervals.append((utterance["start"], utterance["end"]))
    george_intervals[key] = intervals

In [59]:
george_intervals[george_files[0]]

[(6720, 11200), (12400, 13520), (16160, 17760), (24000, 24480)]

In [68]:
from pathlib import Path
from pydub import AudioSegment

def extract_and_concat_segments(
    input_path: str | Path,
    segments_ms: list[tuple[int, int]],
    output_path: str | Path,
    output_format: str | None = None,  # e.g. "wav", "mp3"; if None, infer from output_path suffix
) -> Path:
    input_path = Path(input_path)
    output_path = Path(output_path)

    if output_format is None:
        output_format = output_path.suffix.lstrip(".").lower() or "wav"

    audio = AudioSegment.from_file(input_path)

    # Build output by concatenating slices
    out = AudioSegment.silent(duration=0, frame_rate=audio.frame_rate)

    for start_ms, end_ms in segments_ms:
        if start_ms < 0 or end_ms < 0:
            raise ValueError(f"Negative times not allowed: {(start_ms, end_ms)}")
        if end_ms <= start_ms:
            continue  # skip empty/invalid segment quietly

        start_ms = min(start_ms, len(audio))
        end_ms = min(end_ms, len(audio))
        if end_ms <= start_ms:
            continue

        out += audio[start_ms:end_ms]

    out.export(output_path, format=output_format)
    return output_path

# Example:
# segments = [(1200, 4500), (9000, 12500), (20000, 23000)]
# extract_and_concat_segments("input.wav", segments, "output.wav")


In [69]:
output_dir = base_dir / "Concat Transactions"
output_dir.mkdir(parents = True, exist_ok = True)

In [73]:
file_name_counter = 1
for file, intervals in julian_intervals.items():
    extract_and_concat_segments(file, intervals, output_dir / f"Julian_concat_{file_name_counter}.wav")
    file_name_counter += 1


In [74]:
file_name_counter = 1
for file, intervals in george_intervals.items():
    extract_and_concat_segments(file, intervals, output_dir / f"George_concat_{file_name_counter}.wav")
    file_name_counter += 1

In [86]:
from pathlib import Path
base_dir = Path.cwd()

#Set up Various Directories
#Julian and George Raw Ground Truths Live in "Raw Ground Truths"
#They will go through processing and move to Processed Ground Truths, then prepare_audio_for_embedding() method and move to "To Be Embedded Ground Truths"
#Transaction Clips live in "Processed Transaction Clips". They only need to go through prepare_audio_for_embedding() method and move to "To Be Embedded Transactions


raw_truth_dir = base_dir / "Raw Ground Truths"
transaction_dir = base_dir / "Concat Transactions"

processed_dir = base_dir / "Processed Ground Truths"
output_dir = base_dir / "To Be Embedded Ground Truths"
to_embed_dir = base_dir / "To Be Embedded Transactions"

processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents = True, exist_ok = True)
to_embed_dir.mkdir(parents = True, exist_ok=True)

In [87]:
#Import Methods from utils.py
from utils import prepare_audio_for_embedding, decode_audio, preprocess_audio, save_audio

In [88]:
#Preprocessing Steps for Raw Ground Truths
#Pass through methods decode_audio(), preprocess_audio(), and save_audio()
#decode_audio() and preprocess_audio() combine to apply bandpass, AGC and limiter (in that order)
#save_audio() will save ground truths to "Processed Ground Truths"
from tqdm import tqdm

raw_input_files = [
    p for p in raw_truth_dir.iterdir()
    if p.is_file() and not p.name.startswith(".")
]
print(f"Found {len(raw_input_files)} raw input files.")

processed_audio_samples = []

for i, file_path in enumerate(raw_input_files):
    file_name = str(raw_input_files[i])[len(str(raw_truth_dir))+1:]
    audio, sr = decode_audio(file_path)
    processed_audio_sample = preprocess_audio(audio, sr)
    output_path = save_audio(audio, sr, processed_dir / f"Processed_{file_name}")
    print(f"Saved file to {output_path}")

Found 16 raw input files.
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Processed Ground Truths/Processed_Oswaldo Ballesteros.wav
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Processed Ground Truths/Processed_Edward Herrera.wav
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Processed Ground Truths/Processed_Katie Carbajal-Mendoza.wav
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Processed Ground Truths/Processed_Owen LaMontagne.wav
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Processed Ground Truths/Processed_George Robbins.wav
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Processed Ground Truths/Processed_Julian Carmona-Munoz.wav
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Processed Ground Truths/Processed_Katherine Campos.wav
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Processed Ground Truths/Proce

In [89]:
#Get Preprocessed Ground Truths Ready for Embedding
#prepare_audio_for_embedding() method will set frame rate of the .wav to 16kHz
processed_files = [f for f in processed_dir.rglob("*") if f.is_file()]

print(f"Found {len(processed_files)} audio files")

for processed_file in tqdm(processed_files):
    processed_path = prepare_audio_for_embedding(processed_file, output_dir)

Found 17 audio files


  0%|          | 0/17 [00:00<?, ?it/s][NeMo W 2026-02-16 18:52:43 nemo_logging:405] /opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/pydub/utils.py:198: RuntimeWarning: Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work
      warn("Couldn't find ffprobe or avprobe - defaulting to ffprobe, but may not work", RuntimeWarning)
    
ERROR:utils:Failed to prepare audio /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Processed Ground Truths/.DS_Store: [Errno 2] No such file or directory: 'ffprobe'
100%|██████████| 17/17 [00:00<00:00, 308.13it/s]


In [90]:
#Get Preprocessed Transaction Clips Ready for Embedding
processed_files = [f for f in transaction_dir.rglob("*") if f.is_file()]

print(f"Found {len(processed_files)} audio files")

for processed_file in tqdm(processed_files):
    processed_path = prepare_audio_for_embedding(processed_file, to_embed_dir)

Found 13 audio files


100%|██████████| 13/13 [00:00<00:00, 450.94it/s]


In [91]:
#Standard Code to set up TitaNet
#Set to eval() mode for future embedding
import torch
import nemo.collections.asr as nemo_asr

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

# Pretrained TitaNet Large speaker verification model
speaker_model = nemo_asr.models.EncDecSpeakerLabelModel.from_pretrained(
    model_name="nvidia/speakerverification_en_titanet_large"
).to(device)

speaker_model.eval()

[NeMo W 2026-02-16 18:52:44 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2026-02-16 18:52:44 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data

device: mps
[NeMo I 2026-02-16 18:52:44 nemo_logging:393] PADDING: 16
[NeMo I 2026-02-16 18:52:44 nemo_logging:393] Model EncDecSpeakerLabelModel was successfully restored from /Users/anshulchiranth/.cache/huggingface/hub/models--nvidia--speakerverification_en_titanet_large/snapshots/0dc382f40121a5fbd34db10a2bb04d826c2be6a8/speakerverification_en_titanet_large.nemo.


EncDecSpeakerLabelModel(
  (loss): AngularSoftmaxLoss()
  (eval_loss): AngularSoftmaxLoss()
  (_accuracy): TopKClassificationAccuracy()
  (preprocessor): AudioToMelSpectrogramPreprocessor(
    (featurizer): FilterbankFeatures()
  )
  (encoder): ConvASREncoder(
    (encoder): Sequential(
      (0): JasperBlock(
        (mconv): ModuleList(
          (0): MaskedConv1d(
            (conv): Conv1d(80, 80, kernel_size=(3,), stride=(1,), padding=(1,), groups=80, bias=False)
          )
          (1): MaskedConv1d(
            (conv): Conv1d(80, 1024, kernel_size=(1,), stride=(1,), bias=False)
          )
          (2): BatchNorm1d(1024, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
          (3): SqueezeExcite(
            (fc): Sequential(
              (0): Linear(in_features=1024, out_features=128, bias=False)
              (1): ReLU(inplace=True)
              (2): Linear(in_features=128, out_features=1024, bias=False)
            )
            (gap): AdaptiveAvgPool1d(

In [92]:
#Process for Embedding Ground Truth Embeddings
#They are saved to Ground Truth Embeddings/

emb_dir = base_dir / "Ground Truth Embeddings"
emb_dir.mkdir(parents = True, exist_ok = True)

wav_files = sorted([p for p in output_dir.rglob("*.wav")])
print("WAV files found:", len(wav_files))
print("example:", wav_files[0] if wav_files else "None")

import numpy as np
import pandas as pd
from tqdm import tqdm

rows = []
all_embs = {}  # {relative_path: embedding_vector}

with torch.no_grad():
    for wav_path in tqdm(wav_files):
        # NeMo provides get_embedding(path_to_wav) for speaker embeddings :contentReference[oaicite:3]{index=3}
        emb = speaker_model.get_embedding(str(wav_path))

        # Normalize output type -> 1D numpy array
        if isinstance(emb, torch.Tensor):
            emb_np = emb.detach().cpu().numpy()
        else:
            emb_np = np.asarray(emb)

        emb_np = np.squeeze(emb_np)  # ensure shape (D,)

        # Save one embedding per file
        rel = wav_path.relative_to(output_dir)
        out_path = emb_dir / rel.with_suffix(".npy")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_path, emb_np)

        rows.append({
            "wav_path": str(wav_path),
            "embedding_path": str(out_path),
            "dim": int(emb_np.shape[0]),
        })
        all_embs[str(rel)] = emb_np

df = pd.DataFrame(rows)
df.to_csv(emb_dir / "index.csv", index=False)

# Also save a single archive for easy loading later
np.savez_compressed(emb_dir / "embeddings.npz", **all_embs)

print("Saved embeddings to:", emb_dir)
print(df.head())

WAV files found: 16
example: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/To Be Embedded Ground Truths/Processed_Brianna Leach-Richardson_converted.wav


100%|██████████| 16/16 [00:03<00:00,  5.32it/s]

Saved embeddings to: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Ground Truth Embeddings
                                            wav_path  \
0  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
1  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
2  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
3  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
4  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   

                                      embedding_path  dim  
0  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
1  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
2  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
3  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
4  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  


In [93]:
#Process for Embedding Transaction
#Saved to Transaction Embeddings/
emb_dir = base_dir / "Transaction Embeddings"
emb_dir.mkdir(parents = True, exist_ok = True)

wav_files = sorted([p for p in to_embed_dir.rglob("*.wav")])
print("WAV files found:", len(wav_files))
print("example:", wav_files[0] if wav_files else "None")

import numpy as np
import pandas as pd
from tqdm import tqdm

rows = []
all_embs = {}  # {relative_path: embedding_vector}

with torch.no_grad():
    for wav_path in tqdm(wav_files):
        # NeMo provides get_embedding(path_to_wav) for speaker embeddings :contentReference[oaicite:3]{index=3}
        emb = speaker_model.get_embedding(str(wav_path))

        # Normalize output type -> 1D numpy array
        if isinstance(emb, torch.Tensor):
            emb_np = emb.detach().cpu().numpy()
        else:
            emb_np = np.asarray(emb)

        emb_np = np.squeeze(emb_np)  # ensure shape (D,)

        # Save one embedding per file
        rel = wav_path.relative_to(to_embed_dir)
        out_path = emb_dir / rel.with_suffix(".npy")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_path, emb_np)

        rows.append({
            "wav_path": str(wav_path),
            "embedding_path": str(out_path),
            "dim": int(emb_np.shape[0]),
        })
        all_embs[str(rel)] = emb_np

df = pd.DataFrame(rows)
df.to_csv(emb_dir / "index.csv", index=False)

# Also save a single archive for easy loading later
np.savez_compressed(emb_dir / "embeddings.npz", **all_embs)

print("Saved embeddings to:", emb_dir)
print(df.head())

WAV files found: 13
example: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/To Be Embedded Transactions/George_concat_1_converted.wav


100%|██████████| 13/13 [00:00<00:00, 17.76it/s]

Saved embeddings to: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transaction Embeddings
                                            wav_path  \
0  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
1  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
2  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
3  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
4  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   

                                      embedding_path  dim  
0  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
1  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
2  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
3  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
4  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  


In [94]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- paths (relative to current working directory) ---
tx_dir = Path.cwd() / "Transaction Embeddings"
gt_dir = Path.cwd() / "Ground Truth Embeddings"

# --- collect only .npy files, ignore everything else ---
tx_files = sorted([p for p in tx_dir.iterdir() if p.is_file() and p.suffix == ".npy"])
gt_files = sorted([p for p in gt_dir.iterdir() if p.is_file() and p.suffix == ".npy"])

if not tx_files:
    raise FileNotFoundError(f"No .npy files found in: {tx_dir}")
if not gt_files:
    raise FileNotFoundError(f"No .npy files found in: {gt_dir}")

# --- load helpers ---
def load_vec_192(path: Path) -> np.ndarray:
    """Load an embedding and return shape (192,) float32."""
    v = np.load(path)
    v = np.asarray(v, dtype=np.float32).reshape(-1)  # flatten any (1,192) or (192,1) etc.
    if v.shape[0] != 192:
        raise ValueError(f"{path.name}: expected 192 values, got {v.shape[0]} with original shape {np.load(path).shape}")
    return v

# --- load all embeddings ---
tx_mat = np.stack([load_vec_192(p) for p in tx_files], axis=0)   # (N, 192)
gt_mat = np.stack([load_vec_192(p) for p in gt_files], axis=0)   # (M, 192)

# --- cosine similarity (manual; no sklearn needed) ---
# sim = (A @ B.T) / (||A|| * ||B||)
tx_norm = np.linalg.norm(tx_mat, axis=1, keepdims=True)          # (N, 1)
gt_norm = np.linalg.norm(gt_mat, axis=1, keepdims=True)          # (M, 1)

# avoid divide-by-zero just in case
tx_norm = np.clip(tx_norm, 1e-12, None)
gt_norm = np.clip(gt_norm, 1e-12, None)

sim = (tx_mat @ gt_mat.T) / (tx_norm * gt_norm.T)               # (N, M)

# --- build dataframe ---
row_names = [p.stem for p in tx_files]
col_names = [p.stem for p in gt_files]

df = pd.DataFrame(sim, index=row_names, columns=col_names).round(5)

# mapping from full column name → operator name

operator_map = {
    'Processed_Brianna Leach-Richardson_converted': "Brianna",
       'Processed_Edward Herrera_converted': "Edward",
       'Processed_George Robbins_converted': "George",
       'Processed_Gunnar Mckinnon_converted': "Gunnar", 'Processed_Jake Wohl_converted': "Jake",
       'Processed_Julian Carmona-Munoz_converted': "Julian",
       'Processed_Katherine Campos_converted': "Katherine",
       'Processed_Katie Carbajal-Mendoza_converted': "Katie",
       'Processed_Kayla Devito_converted': "Kayla",
       'Processed_Kennith Newkirk_converted': "Kennith",
       'Processed_Kyleigh Harp_converted': "Kyleigh", 'Processed_Maria Mendoza_converted': "Maria",
       'Processed_Olivia Jensen_converted': "Olivia",
       'Processed_Oswaldo Ballesteros_converted': "Oswaldo",
       'Processed_Owen LaMontagne_converted': "Owen",
       'Processed_Shaniyah Smith_converted': "Shaniyah"
    
}





"""
operator_map = {
    "Cosine Similarity (George)": "George",
    "Cosine Similarity (Julian)": "Julian"
}
"""

# Get column name of max similarity per row
max_cols = df.idxmax(axis=1)

# Map to clean operator names
df["Recognized Operator"] = max_cols.map(operator_map)

# Convert index to a Series so we can use .str methods
index_series = df.index.to_series()

df["Ground Truth Operator"] = None  # initialize column

df.loc[index_series.str.contains("George", case=False, na=False),
       "Ground Truth Operator"] = "George"

df.loc[index_series.str.contains("Julian", case=False, na=False),
       "Ground Truth Operator"] = "Julian"

df["Correct Identification"] = (
    df["Recognized Operator"] == df["Ground Truth Operator"]
)

In [95]:
df

,Processed_Brianna Leach-Richardson_converted,Processed_Edward Herrera_converted,Processed_George Robbins_converted,Processed_Gunnar Mckinnon_converted,Processed_Jake Wohl_converted,Processed_Julian Carmona-Munoz_converted,Processed_Katherine Campos_converted,Processed_Katie Carbajal-Mendoza_converted,Processed_Kayla Devito_converted,Processed_Kennith Newkirk_converted,Processed_Kyleigh Harp_converted,Processed_Maria Mendoza_converted,Processed_Olivia Jensen_converted,Processed_Oswaldo Ballesteros_converted,Processed_Owen LaMontagne_converted,Processed_Shaniyah Smith_converted,Recognized Operator,Ground Truth Operator,Correct Identification
George_concat_1_converted,0.12726,0.07874,0.75577,0.08926,0.05106,0.14655,0.20752,0.05417,0.05795,0.03568,0.16277,0.08972,0.16716,0.03024,0.20905,0.10784,George,George,True
George_concat_2_converted,0.13037,0.13677,0.82918,0.09223,0.07527,0.16106,0.19435,0.08674,0.02812,0.09012,0.16543,0.12574,0.20217,0.07184,0.24838,0.11913,George,George,True
George_concat_3_converted,0.09874,-0.01228,0.68527,0.08962,0.01632,0.15262,0.16444,0.02111,0.06273,0.01783,0.13178,0.10959,0.16791,-0.07898,0.20951,0.15990,George,George,True
George_concat_4_converted,0.14963,0.05216,0.77216,0.08562,0.02955,0.16175,0.21764,0.05404,0.05401,-0.02145,0.18835,0.14511,0.17860,-0.10390,0.17899,0.17038,George,George,True
George_concat_5_converted,0.15097,0.08782,0.78196,0.09078,0.11487,0.19637,0.22156,0.06363,0.03747,0.04185,0.17342,0.10306,0.18204,-0.08747,0.21046,0.17670,George,George,True
George_concat_6_converted,0.16205,0.17867,0.81675,0.08211,0.01846,0.12295,0.08395,0.04168,0.08084,-0.01344,0.09203,0.12491,0.15704,-0.03435,0.20711,0.08187,George,George,True
Julian_concat_1_converted,0.04141,0.28624,0.12366,0.10780,0.26800,0.75461,0.24800,0.08918,-0.00406,0.06539,0.12738,0.11862,0.09485,0.20372,0.28184,0.15654,Julian,Julian,True
Julian_concat_2_converted,0.08114,0.31546,0.12333,0.11069,0.32697,0.79004,0.19087,0.12345,-0.02374,0.07121,0.10458,0.11854,0.06994,0.25602,0.24186,0.17057,Julian,Julian,True
Julian_concat_3_converted,0.01368,0.25088,0.15550,0.08043,0.22530,0.67263,0.18439,0.12736,-0.03778,0.09939,0.11536,0.13085,0.06463,0.18690,0.25973,0.14081,Julian,Julian,True
Julian_concat_4_converted,0.08455,0.36684,-0.00691,0.06771,0.37104,0.79566,0.18720,0.14494,0.05199,0.15419,0.14649,0.16688,0.11880,0.28975,0.21982,0.12493,Julian,Julian,True


In [96]:
accuracy = (len(df[df["Correct Identification"] == True]) / len(df) ) * 100
print(f"Final Accuracy for binary classification task is: {accuracy}%")

Final Accuracy for binary classification task is: 100.0%


In [ ]:
df[[]]